# 07 Trigger agent case (demo operator, SYNTHETIC data)

Starts the coworker agent for one style x market without the Azure CLI or a function key. The notebook writes the
same JSON event that Activator's relay `udf_agent_relay.relay_t1` writes to `lh_medallion` `Files/agent_outbox/<UTC date>/`.
The agent Function (`outbox_poll`) reads that folder every minute, re-checks the signal, and opens the case in Teams.

**Use:** set the parameters below (default HM9443 / US) and select **Run all**. A Teams case chat appears in about 1-2 minutes.

* If a case for the same style x market is still open, no second case is opened. The open chat gets a "New signal"
  update instead (at most once per hour). Close the old case first with Databricks notebook `05_demo_reset` (`action = close_case`).
* The agent opens a case only if the T1 stock-out risk is confirmed by the data; otherwise it logs `AA-00_NOT_VERIFIED` and does nothing.
* Each run writes a new file, and the agent claims each file once.

In [ ]:
# Parameters
style_id = "HM9443"     # Men's AeroGlide 5 (SYNTHETIC)
market_id = "US"

In [ ]:
import datetime as dt
import json
import os
import re

OUTBOX = "/lakehouse/default/Files/agent_outbox"   # default lakehouse = lh_medallion
style, market = style_id.strip().upper(), market_id.strip().upper()
if not re.fullmatch(r"[A-Z0-9][A-Z0-9-]{1,31}", style) or not re.fullmatch(r"[A-Z]{2,8}", market):
    raise ValueError("style_id and market_id must be catalogue codes, e.g. HM9443 / US")
if not os.path.isdir("/lakehouse/default/Files"):
    raise RuntimeError("Attach lh_medallion as the default lakehouse of this notebook")

now = dt.datetime.now(dt.timezone.utc).replace(microsecond=0)
folder = f"{OUTBOX}/{now:%Y-%m-%d}"   # the agent scans today's and yesterday's UTC folders
name = f"{now:%Y%m%dT%H%M%SZ}_{style}_{market}_manual.json"
event = {"style_id": style, "market_id": market, "source": "manual-notebook",
         "rule": "Manual demo trigger (07_trigger_agent_case)", "written_at": f"{now:%Y-%m-%dT%H:%M:%SZ}",
         "record_label": "SYNTHETIC"}
os.makedirs(folder, exist_ok=True)
with open(f"{folder}/{name}", "w", encoding="utf-8") as f:
    json.dump(event, f)
print(f"Queued {style} / {market} for the coworker agent: Files/agent_outbox/{now:%Y-%m-%d}/{name}")
print("Watch Teams for the case chat (about 1-2 minutes).")